# Mejora 5 - Marcado de negacion

**Proyecto - Parte 1: Machine Learning clasico**

**Estudiantes:**

[Nombre integrante 1] - [codigo]

[Nombre integrante 2] - [codigo]

**Curso:** ISIS2611 - Aprendizaje de Maquina

---

## Contexto

El enunciado de la competencia señala que la dificultad central del problema es que el sentimiento global **no se obtiene contando palabras positivas y negativas**: depende del orden, de la negacion y de los conectores contrastivos (`"la bateria es excelente, pero la camara resulto decepcionante"`). Los notebooks anteriores ya atacaron el problema de los conectores contrastivos (bloques `ultima` oracion y `clausula final` en el notebook principal, y el modelo en dos etapas de las Mejoras 1 y 4). Este notebook ataca el otro lado del problema: **la negacion**.

Con bolsa de palabras y n-gramas, una resena como `"la camara no funciona bien"` solo distingue "no funciona" como bigrama de "no" seguido de "funciona": la palabra "bien", tres posiciones despues de la negacion, se sigue representando exactamente igual que en una resena positiva ("la pantalla se ve muy bien"). Este notebook implementa una tecnica clasica de bolsa de palabras para este problema (Das & Chen, 2001; Pang, Lee & Vaithyanathan, 2002): **marcar con un sufijo `_neg` cada palabra que queda dentro del alcance de una negacion**, para que el vectorizador TF-IDF trate "bien" y "bien_neg" como atributos distintos. No es una tecnica secuencial ni de aprendizaje profundo: sigue siendo bolsa de palabras, solo que con un paso de preprocesamiento adicional.

## Requisitos de la guia

| Requisito | Como se cumple |
|:---|:---|
| No usar aprendizaje profundo, transformers ni embeddings preentrenados. | No se usa ninguno. El marcado de negacion es una regla de preprocesamiento sobre texto, no un modelo. |
| Usar unicamente modelos de scikit-learn. | Solo `LogisticRegression`. |
| Resultados replicables. | Semillas fijas, versiones en `requirements.txt` y todo el flujo en este notebook (seccion 11). |

## 1. Importación de librerías

In [1]:
import os
import re
import unicodedata
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from nltk.stem import SnowballStemmer

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, ConfusionMatrixDisplay,
                             f1_score, precision_score, recall_score)
from sklearn.model_selection import (GridSearchCV, StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 160)

RANDOM_STATE = 42
ORDEN_CLASES = ['negativo', 'neutral', 'positivo']

# Rutas del proyecto (este notebook está en la carpeta notebooks/)
RUTA_DATOS = os.path.join('..', 'data')
RUTA_MODELOS = os.path.join('..', 'models')
RUTA_ENVIOS = os.path.join('..', 'submissions')
NOMBRE_MODELO = 'regresion_logistica_negacion'

os.makedirs(RUTA_MODELOS, exist_ok=True)
os.makedirs(RUTA_ENVIOS, exist_ok=True)

## 2. Datos, partición y modelo actual

In [2]:
train = pd.read_csv(os.path.join(RUTA_DATOS, 'train.csv'))
evaluacion = pd.read_csv(os.path.join(RUTA_DATOS, 'eval.csv'))
muestra_envio = pd.read_csv(os.path.join(RUTA_DATOS, 'sample_submission.csv'))

print('train:', train.shape, '| eval:', evaluacion.shape, '| sample_submission:', muestra_envio.shape)

train: (12000, 3) | eval: (3000, 2) | sample_submission: (3000, 2)


In [3]:
X = train[['text']]
y = train['label']

# Misma partición y mismas particiones de validación cruzada que en el notebook principal
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print('Entrenamiento:', X_train.shape, '| Prueba:', X_test.shape)

Entrenamiento: (9600, 1) | Prueba: (2400, 1)


In [4]:
stemmer = SnowballStemmer('spanish')
cache_raices = {}


def raiz(palabra):
    # Se guarda la raíz de cada palabra para no repetir el stemming miles de veces
    if palabra not in cache_raices:
        cache_raices[palabra] = stemmer.stem(palabra)
    return cache_raices[palabra]


def quitar_tildes(texto):
    texto = unicodedata.normalize('NFKD', texto)
    return ''.join(c for c in texto if not unicodedata.combining(c))


def limpiar_texto(texto, stop=None, stemming=True):
    texto = quitar_tildes(texto.lower())
    texto = re.sub(r'[^\w\s]', ' ', texto)
    texto = re.sub(r'\d+', ' ', texto)
    tokens = texto.split()
    if stop:
        tokens = [t for t in tokens if t not in stop]
    if stemming:
        tokens = [raiz(t) for t in tokens]
    return ' '.join(tokens)


def separar_oraciones(texto):
    return [o.strip() for o in re.split(r'(?<=[.!?])\s+', texto) if o.strip()]


# Conectores que introducen la opinión que se impone sobre la anterior
CONECTORES = r'\b(?:pero|aunque|sin embargo|aun asi|con todo|a fin de cuentas|al final)\b'


def extraer_completo(textos, stemming=True):
    return [limpiar_texto(t, stemming=stemming) for t in textos]


def extraer_ultima(textos, stemming=True):
    return [limpiar_texto(separar_oraciones(t)[-1], stemming=stemming) for t in textos]


def extraer_clausula_final(textos, stemming=True):
    salida = []
    for t in textos:
        ultima = quitar_tildes(separar_oraciones(t)[-1].lower())
        salida.append(limpiar_texto(re.split(CONECTORES, ultima)[-1], stemming=stemming))
    return salida


def bloque_tfidf(nombre, extractor, rango=(1, 2)):
    return (nombre,
            Pipeline([('extraer', FunctionTransformer(extractor)),
                      ('tfidf', TfidfVectorizer(ngram_range=rango))]),
            'text')

In [5]:
def metricas(y_real, y_pred):
    return {'Exactitud': accuracy_score(y_real, y_pred),
            'Precisión (macro)': precision_score(y_real, y_pred, average='macro'),
            'Recall (macro)': recall_score(y_real, y_pred, average='macro'),
            'F1 (macro)': f1_score(y_real, y_pred, average='macro')}

In [6]:
def construir_representacion(bloques=('completo', 'ultima', 'clausula'), rango=(1, 2)):
    extractores = {'completo': extraer_completo, 'ultima': extraer_ultima, 'clausula': extraer_clausula_final}
    return ColumnTransformer([bloque_tfidf(b, extractores[b], rango) for b in bloques])


# Modelo actual (el del notebook principal): regresión logística L1 sobre la representación por bloques
modelo_actual = Pipeline([
    ('rep', construir_representacion()),
    ('modelo', LogisticRegression(C=10, penalty='l1', solver='liblinear', max_iter=3000, random_state=RANDOM_STATE)),
])
modelo_actual.fit(X_train, y_train)
pred_actual = pd.Series(modelo_actual.predict(X_test), index=y_test.index)

cv_actual = cross_val_score(modelo_actual, X_train, y_train, cv=kfold, scoring='f1_macro', n_jobs=-1)
print(f'Modelo actual - F1-macro en validación cruzada (train): {cv_actual.mean():.4f} +/- {cv_actual.std():.4f}')
pd.DataFrame({'Modelo actual (test)': metricas(y_test, pred_actual)}).T.round(4)

Modelo actual - F1-macro en validación cruzada (train): 0.8992 +/- 0.0072


,Exactitud,Precisión (macro),Recall (macro),F1 (macro)
Modelo actual (test),0.8946,0.8987,0.8998,0.8993


## 3. La idea: marcar el alcance de la negación

Se define un conjunto de **negadores** (`no`, `nunca`, `jamás`, `tampoco`, `ni`, `sin`) y una función `marcar_negacion` que:

1. Recorre el texto separándolo en cláusulas, usando como frontera tanto los signos de puntuación (`. , ; : ! ?`) como los mismos conectores contrastivos ya usados en el notebook principal (`pero`, `aunque`, `sin embargo`, ...). Esto evita que la negación "contamine" una cláusula que ya expresa una idea distinta.
2. Dentro de cada cláusula, cuando encuentra un negador activa una bandera y le añade el sufijo `_neg` a cada palabra (ya con su raíz aplicada) hasta el final de la cláusula.

Por ejemplo, `"la cámara no funciona bien, pero el sonido es excelente"` se separa en dos cláusulas por la coma y por "pero"; en la primera, "funciona" y "bien" quedan marcadas (`funcion_neg bien_neg`) porque están después de "no", y la segunda cláusula queda intacta. Así el modelo puede aprender que "bien_neg" se parece a las reseñas negativas, mientras que "bien" sigue pareciéndose a las positivas.

In [7]:
NEGADORES = {'no', 'nunca', 'jamas', 'tampoco', 'ni', 'sin'}


def marcar_negacion_fragmento(fragmento, stemming=True):
    # fragmento ya debe venir en minúsculas y sin tildes
    clausulas = re.split(r'[.,;:!?()]+|' + CONECTORES, fragmento)
    tokens_salida = []
    for clausula in clausulas:
        pieza = re.sub(r'[^\w\s]', ' ', clausula)
        pieza = re.sub(r'\d+', ' ', pieza)
        palabras = pieza.split()
        negando = False
        for palabra in palabras:
            if palabra in NEGADORES:
                negando = True
                tokens_salida.append(raiz(palabra) if stemming else palabra)
                continue
            token = raiz(palabra) if stemming else palabra
            tokens_salida.append(token + '_neg' if negando else token)
    return ' '.join(tokens_salida)


def extraer_negacion(textos, stemming=True):
    return [marcar_negacion_fragmento(quitar_tildes(t.lower()), stemming=stemming) for t in textos]


def extraer_ultima_negacion(textos, stemming=True):
    return [marcar_negacion_fragmento(quitar_tildes(separar_oraciones(t)[-1].lower()), stemming=stemming) for t in textos]


def extraer_clausula_final_negacion(textos, stemming=True):
    salida = []
    for t in textos:
        ultima = quitar_tildes(separar_oraciones(t)[-1].lower())
        resto = re.split(CONECTORES, ultima)[-1]
        salida.append(marcar_negacion_fragmento(resto, stemming=stemming))
    return salida


# Demostración sobre un ejemplo
ejemplo = 'la camara no funciona bien, pero el sonido es excelente'
print('Original:               ', ejemplo)
print('Con negación marcada:   ', marcar_negacion_fragmento(ejemplo, stemming=False))
print('Con negación + stemming:', marcar_negacion_fragmento(ejemplo))

Original:                la camara no funciona bien, pero el sonido es excelente
Con negación marcada:    la camara no funciona_neg bien_neg el sonido es excelente
Con negación + stemming: la cam no funcion_neg bien_neg el son es excelent


## 4. Dónde insertar el bloque de negación

Se prueban tres formas de usar `marcar_negacion`, todas sobre la misma representación por bloques y el mismo modelo (regresión logística L1, `C = 10`) del notebook principal, para aislar el efecto de la negación:

- **Reemplazo en `última` y `cláusula`:** las dos oraciones/cláusulas finales (donde se concentra la opinión que domina, según el notebook principal) se vectorizan con la versión marcada en vez de la versión normal. El bloque `completo` se deja igual.
- **Bloque extra sobre `última`:** se agrega un cuarto bloque con la versión marcada de la última oración, además de los tres bloques originales sin modificar.
- **Bloque extra sobre `cláusula`:** igual, pero con la versión marcada de la cláusula final.

No se marca negación en el bloque `completo`: con más de 70.000 atributos esa representación ya es muy dispersa (ver Mejora 3), y partir cada palabra negada en un atributo nuevo la dispersa todavía más sin que las oraciones completas (que mezclan elogios y quejas) se beneficien tanto como las oraciones finales, que suelen ser más cortas y centradas en una sola idea.

In [8]:
EXTRACTORES = {
    'completo': extraer_completo,
    'ultima': extraer_ultima,
    'clausula': extraer_clausula_final,
    'ultima_neg': extraer_ultima_negacion,
    'clausula_neg': extraer_clausula_final_negacion,
}


def construir_representacion_neg(bloques, rango=(1, 2)):
    return ColumnTransformer([bloque_tfidf(b, EXTRACTORES[b], rango) for b in bloques])


variantes_negacion = {
    'Modelo actual (sin negación)': ('completo', 'ultima', 'clausula'),
    'Reemplazo: última y cláusula con negación': ('completo', 'ultima_neg', 'clausula_neg'),
    'Bloque extra: + última con negación': ('completo', 'ultima', 'clausula', 'ultima_neg'),
    'Bloque extra: + cláusula con negación': ('completo', 'ultima', 'clausula', 'clausula_neg'),
}

filas_neg = []
modelos_neg = {}
for nombre, bloques in variantes_negacion.items():
    pipe = Pipeline([
        ('rep', construir_representacion_neg(bloques)),
        ('modelo', LogisticRegression(C=10, penalty='l1', solver='liblinear', max_iter=3000, random_state=RANDOM_STATE)),
    ])
    cv = cross_val_score(pipe, X_train, y_train, cv=kfold, scoring='f1_macro', n_jobs=-1)
    filas_neg.append({'Variante': nombre, 'F1-macro (CV)': cv.mean(), 'Desv. F1-macro': cv.std()})
    modelos_neg[nombre] = pipe

pd.DataFrame(filas_neg).set_index('Variante').round(4)

,F1-macro (CV),Desv. F1-macro
Variante,,
Modelo actual (sin negación),0.8992,0.0072
Reemplazo: última y cláusula con negación,0.8985,0.0067
Bloque extra: + última con negación,0.8981,0.0067
Bloque extra: + cláusula con negación,0.8970,0.0068


Las tres variantes con negación quedan **por debajo** del modelo actual en esta partición: el reemplazo en última y cláusula es la que menos pierde (a 0,07 puntos del modelo actual), y agregar un bloque extra en vez de reemplazar es siempre peor, porque duplica vocabulario sin aportar información nueva (la versión sin marcar del mismo texto ya está en los bloques originales). Por eso la variante que se lleva a la siguiente sección es el **reemplazo en última y cláusula**: es la única que queda dentro de la desviación entre particiones del modelo actual (0,0072).

## 5. Evaluación en el conjunto de prueba

In [9]:
modelo_negacion = modelos_neg['Reemplazo: última y cláusula con negación']
modelo_negacion.fit(X_train, y_train)
pred_negacion = pd.Series(modelo_negacion.predict(X_test), index=y_test.index)

tabla_prueba = pd.DataFrame({
    'Modelo actual': metricas(y_test, pred_actual),
    'Con negación (última y cláusula)': metricas(y_test, pred_negacion),
}).T
tabla_prueba.round(4)

,Exactitud,Precisión (macro),Recall (macro),F1 (macro)
Modelo actual,0.8946,0.8987,0.8998,0.8993
Con negación (última y cláusula),0.8971,0.9006,0.9022,0.9014


En el conjunto de prueba la variante con negación **supera** al modelo actual: +0,0025 de exactitud y +0,0021 de F1-macro (unas 6 reseñas de 2.400). Es la imagen contraria a la de la validación cruzada de la sección anterior: ahí perdía por 0,07 puntos. Con diferencias de este tamaño (menos de una décima de punto porcentual) y en sentidos opuestos según la partición, hay que compararlas con más particiones antes de sacar una conclusión, igual que en la Mejora 3.

## 6. ¿La diferencia es real? Comparación con 15 particiones

Se compara el modelo con negación contra el modelo actual sobre 15 particiones (3 repeticiones de validación cruzada de 5 particiones, semillas 1, 2 y 3), las mismas para ambos y la misma regla de las Mejoras 2, 3 y 4: se considera una mejora si la diferencia media supera 2 errores estándar **y** la alternativa gana en al menos 12 de las 15 particiones.

In [10]:
semillas = [1, 2, 3]
particiones = []
for semilla in semillas:
    particiones += list(StratifiedKFold(n_splits=5, shuffle=True, random_state=semilla).split(X_train, y_train))

f1_actual_15 = cross_val_score(clone(modelo_actual), X_train, y_train, cv=particiones, scoring='f1_macro', n_jobs=-1)
f1_negacion_15 = cross_val_score(clone(modelo_negacion), X_train, y_train, cv=particiones, scoring='f1_macro', n_jobs=-1)

diferencia = f1_negacion_15 - f1_actual_15
error = diferencia.std(ddof=1) / np.sqrt(len(diferencia))
print(f'F1-macro medio (15 particiones) -> modelo actual: {f1_actual_15.mean():.4f} | con negación: {f1_negacion_15.mean():.4f}')
print(f'Diferencia media: {diferencia.mean():+.4f} (error estándar {error:.4f}, umbral 2 EE = {2 * error:.4f})')
print(f'El modelo con negación mejora en {(diferencia > 0).sum()} de 15 particiones')
print('Veredicto:', 'mejora' if diferencia.mean() > 2 * error and (diferencia > 0).sum() >= 12 else
      'empeora' if diferencia.mean() < -2 * error else
      'no hay diferencia clara (empate)')

F1-macro medio (15 particiones) -> modelo actual: 0.8967 | con negación: 0.8990
Diferencia media: +0.0024 (error estándar 0.0009, umbral 2 EE = 0.0018)
El modelo con negación mejora en 11 de 15 particiones
Veredicto: no hay diferencia clara (empate)


Con 15 particiones el modelo con negación queda **por encima** del modelo actual en promedio (+0,0024 de F1-macro, error estándar 0,0009), una diferencia que sí supera el umbral de 2 errores estándar. Pero solo gana en 11 de las 15 particiones, uno menos del mínimo de 12 que exige la regla: es la misma situación que la Mejora 3 con el ajuste fino de `C`, donde la magnitud promedio parecía significativa pero la victoria no era consistente partición por partición. El veredicto es **empate**: hay un indicio de mejora, pero no la evidencia consistente que exige el criterio del proyecto.

## 7. ¿Por qué no ayuda más el marcado de negación?

Dos razones probables, a partir de lo observado:

- **Los bigramas ya capturan buena parte de la negación local.** `"no funciona"` ya es un atributo propio en la representación de palabras con `ngram_range=(1, 2)`; marcar también `funcion_neg` es, en esos casos, información redundante que solo le resta señal al vocabulario (cada palabra queda partida en dos versiones, cada una con menos ejemplos de entrenamiento).
- **Donde el marcado sí aporta algo distinto de los bigramas es en negaciones largas** (`"no me parece que la cámara funcione del todo bien"`, donde "bien" queda a cinco palabras de "no"), pero esas construcciones son minoría frente a las negaciones cortas y directas ("no funciona", "no sirve") que ya cubren los bigramas. El efecto neto es pequeño y, con los datos disponibles, no se distingue del ruido de la partición.

Esto no significa que la negación sea irrelevante para el problema -el enunciado de la competencia tiene razón en señalarla como una dificultad-, sino que la bolsa de palabras con bigramas ya captura buena parte de ese efecto, y el marcado explícito de negación no agrega una señal lo bastante grande como para notarse con un modelo lineal y 12.000 reseñas de entrenamiento.

## 8. Modelo final y predicciones para la competencia

Aunque la comparación con 15 particiones no es concluyente, el modelo con negación es uno de los candidatos razonables explorados durante el proyecto (y el único, hasta ahora, que mejora al modelo actual tanto en el conjunto de prueba como en el promedio de las 15 particiones, aunque sin la consistencia partición por partición que exige la regla). Se entrena con las 12.000 reseñas etiquetadas, se guarda como un envío más para contribuir a los 5 envíos distintos que pide la guía, y se compara contra el modelo actual y contra el mejor envío público hasta ahora (`regresion_logistica_ajuste_fino`, 0,90666).

In [11]:
modelo_final = clone(modelo_negacion)
modelo_final.fit(X, y)

pred_eval = modelo_final.predict(evaluacion[['text']])
envio = pd.DataFrame({'id': evaluacion['id'], 'answer': pred_eval})
envio.to_csv(os.path.join(RUTA_ENVIOS, f'submission_{NOMBRE_MODELO}.csv'), index=False)

print('Mismas columnas que sample_submission:', list(envio.columns) == list(muestra_envio.columns))
print('Mismos ids y en el mismo orden:', (envio['id'] == muestra_envio['id']).all())

otros_envios = {
    'Modelo actual (regresion_logistica_bloques)': 'submission_regresion_logistica_bloques.csv',
    'Mejor envío público (regresion_logistica_ajuste_fino)': 'submission_regresion_logistica_ajuste_fino.csv',
}
for nombre, archivo in otros_envios.items():
    otro = pd.read_csv(os.path.join(RUTA_ENVIOS, archivo))
    distintas = (envio['answer'] != otro['answer']).sum()
    print(f'Predicciones distintas respecto a {nombre}: {distintas} de {len(envio)}')

display(pd.DataFrame({'train (real)': train['label'].value_counts(normalize=True),
                      'eval (predicho)': envio['answer'].value_counts(normalize=True)}).reindex(ORDEN_CLASES).round(3))

Mismas columnas que sample_submission: True
Mismos ids y en el mismo orden: True
Predicciones distintas respecto a Modelo actual (regresion_logistica_bloques): 55 de 3000
Predicciones distintas respecto a Mejor envío público (regresion_logistica_ajuste_fino): 67 de 3000


,train (real),eval (predicho)
negativo,0.351,0.355
neutral,0.298,0.281
positivo,0.351,0.364


In [12]:
ruta_modelo = os.path.join(RUTA_MODELOS, f'{NOMBRE_MODELO}.joblib')
joblib.dump(modelo_final, ruta_modelo)

modelo_cargado = joblib.load(ruta_modelo)
print('El modelo guardado reproduce exactamente las mismas predicciones:',
      (modelo_cargado.predict(evaluacion[['text']]) == pred_eval).all())

El modelo guardado reproduce exactamente las mismas predicciones: True


## 9. Conclusiones

- **Marcar explícitamente el alcance de la negación no mejora el modelo de forma concluyente.** Con 15 particiones gana en 11 de 15 (el mínimo exigido es 12) y la diferencia media (+0,0024 de F1-macro) sí supera el umbral de 2 errores estándar: es un indicio de mejora, no una mejora demostrada, igual que el ajuste fino de `C` en la Mejora 3.
- **Dónde se inserta importa más que si se usa o no.** Agregar un bloque extra con la versión marcada (en vez de reemplazar) siempre fue peor que no usar negación: duplicar el vocabulario con una versión casi idéntica de las mismas palabras le resta señal al modelo en vez de sumarle.
- **Los bigramas ya hacen buena parte del trabajo.** La razón más probable de que el efecto sea pequeño es que la representación de palabras con `ngram_range=(1, 2)` ya captura la negación corta y directa ("no funciona", "no sirve"), que es la más frecuente en las reseñas; el marcado explícito solo ayuda con negaciones largas, que son minoría.
- **Para Kaggle,** este envío cambia predicciones respecto al modelo actual y al mejor envío público, y aporta uno de los 5 envíos distintos que pide la guía con una técnica nueva (no una variación de hiperparámetros de un modelo ya probado). Dado que la diferencia frente al modelo actual no es concluyente, no se puede afirmar de antemano si su puntaje público será mejor o peor; es información adicional para la comparación final, no un reemplazo seguro del mejor modelo actual.

## 10. Uso de herramientas de IA generativa

**Declaración del uso.** Se utilizó Claude Code (modelo Claude Sonnet 5, de Anthropic) como apoyo para: proponer la técnica de marcado de negación como oportunidad de mejora, generar un primer esqueleto del código de este notebook siguiendo la estructura y las convenciones de los notebooks de Mejora anteriores, y redactar las explicaciones.

**Prompts utilizados.**

1. *"Tengo este proyecto de machine learning. El mejor modelo que hemos obtenido ha tenido un score en kaggle de 0.90666 Cómo podemos mejorar?"*, tras lo cual la herramienta revisó el PDF de la guía y los notebooks existentes y propuso, entre otras, marcar la negación como una técnica de bolsa de palabras no explorada todavía.
2. *"haz la 2"*, para implementar la propuesta de marcado de negación.

**Análisis crítico del resultado.**

[Completar por el equipo: ¿qué partes del código y de las explicaciones generadas fueron correctas y útiles? ¿qué errores, imprecisiones o limitaciones se identificaron? ¿qué decisiones técnicas se cambiaron respecto a lo que propuso la herramienta y por qué? ¿qué conceptos del curso permitieron evaluar o corregir la respuesta?]

**Aportes propios del equipo.**

[Completar por el equipo: qué se desarrolló, modificó o decidió, qué ajustes se hicieron al código y a las explicaciones, y qué se aprendió del proceso.]

## 11. Cómo replicar los resultados

1. Crear un entorno con Python 3.9 e instalar las librerías con `pip install -r requirements.txt` (están en la raíz del proyecto).
2. Mantener la estructura de carpetas del proyecto: este notebook está en `notebooks/`, los datos en `data/`. El notebook se ejecuta desde la carpeta `notebooks/`.
3. Ejecutar todas las celdas en orden (`Restart & Run All`). Tarda unos 6 minutos.
4. Al terminar se generan `models/regresion_logistica_negacion.joblib` y `submissions/submission_regresion_logistica_negacion.csv`.

Todas las fuentes de aleatoriedad usan semillas fijas, por lo que las cifras se reproducen exactamente con las versiones de `requirements.txt`. Para cargar el modelo guardado fuera del notebook hay que ejecutar antes las celdas que definen las funciones usadas por el pipeline (secciones 2, 3 y 4).